# Required Python Packages

Before running the code, make sure the following Python packages are installed in a venv. These are necessary for data processing, plotting, and peak detection. Once installed comment them out with '#' as this is only required once


In [160]:
# %pip install pandas
# %pip install numpy
# %pip install matplotlib
# %pip install scipy
# %pip install ipympl
# %pip install PyQt5
# %pip install ruptures
# %pip install openpyxl
# %pip install plotly
# %pip install kaleido

# Peak Detection and Smoothing

This script is used to detect peaks in data, smooth the data, and handle file operations.

### 1. **Library Imports**

- **`scipy.signal`**: Used for signal processing (peak detection, smoothing).
- **`glob` and `os`**: For file handling and navigation.
- **`pandas` (`pd`)**: For data processing, particularly with CSV files.
- **`numpy` (`np`)**: For numerical operations.
- **`math`**: Provides basic mathematical functions.
- **`matplotlib`**: Used for creating plots. The `Qt5Agg` backend allows for interactive plots.


In [161]:
from pathlib import Path
# import kaleido
from datetime import datetime
import plotly.graph_objects as go
from scipy.signal import find_peaks, argrelextrema, savgol_filter, detrend
from glob import glob
import os
import pandas as pd
import numpy as np 
import math  
import matplotlib
matplotlib.use('Qt5Agg')  
import matplotlib.pyplot as plt
import openpyxl

# Dynamic Bin Calculation
This function calculates the number of bins dynamically for histogram plotting using a logarithmic scale.

In [162]:
def calculate_dynamic_bins(n, b_ref=600, t_ref=1200, k=5):
    """
    Dynamically calculate the number of bins for the histogram using logarithmic scaling.
    """
    bins = int(b_ref * math.log(1 + (n * k / t_ref)))
    return min(n, bins)

# Identifying Row with Specific Keyword

This helper function locates the row number in a file where a specified keyword, such as "Time," first appears.


In [163]:
def find_skip_rows(data, keyword='Time'):
    for i, line in enumerate(data):
        if line.strip() and line.strip().startswith(keyword):
            return i
    return -1  # Return -1 if the keyword is not found

# Reading and Processing Behaviour File
This function reads a CSV file and processes the behavior data, organizing it into a dictionary.

In [164]:
def read_and_process_behaviour_file(file_path):
    """Read the CSV file and process its contents."""
    df = pd.read_csv(file_path)
    df.columns = map(str.lower, df.columns)

    # Find the columns of interest
    try:
        behaviour_column = df.columns[df.columns.str.contains(r'behavior')][0]
        start_time_column = df.columns[df.columns.str.contains(
            r'start \(s\)')][0]
        end_time_column = df.columns[df.columns.str.contains(r'stop \(s\)')][0]
        duration_column = df.columns[df.columns.str.contains(
            r'duration \(s\)')][0]
    except IndexError:
        raise ValueError("Required columns are missing in the CSV file")

    # Dictionary to store behavior with start, end, duration, and instance number
    behaviour_data = {}
    for index, row in df.iterrows():
        behaviour = row[behaviour_column]
        start_time = row[start_time_column]
        end_time = row[end_time_column]
        duration = row[duration_column]

        if behaviour not in behaviour_data:
            behaviour_data[behaviour] = []

        instance_number = len(behaviour_data[behaviour]) + 1
        behaviour_data[behaviour].append(
            (instance_number, start_time, end_time, duration))

    return behaviour_data

# File Handling Functions
These functions help read files, process them, and list available files in a directory.

In [165]:
def read_and_process_file(file_path):
    try:
        data = pd.read_csv(file_path, sep='\t', header=None)
        return data
    except Exception as e:
        print(f'''Failed to read the file at {file_path}. Error: {e}''')
        return None

def list_files(directory, extension='csv'):
    print("Listing files in the specified directory:")
    files = glob(directory + '/*' + extension)
    for file in files:
        print(file)

# File Loading and Timestamp Alignment

This section of the code checks if the specified data and behavior file paths are provided and attempts to load the files. If a file can't be found or read, it lists the available files in the directory to help you locate the correct one.

### Key Steps:
- **Data File Check**: 
  - If the `data_file_path` is provided, it attempts to read the file.
  - If the file can't be read, the code lists available files in the directory.
  
- **Behavior File Check**: 
  - Similar to the data file, it checks and reads the `behaviour_file_path`.
  - Lists available files if the behavior file can't be found.
  
- **Timestamp Alignment**: 
  - The provided `probe_date_time` is converted into a timestamp format, which is used later to align the data to a reference point.

After this step, you should see confirmation messages indicating whether the files were successfully read, and the reference timestamp will be printed for alignment purposes.


In [166]:
# Check if the inputs are provided correctly to find the file, read the file, and get the timestamp for alignment
def retireve_data():
    data = None
    if data_file_path:
        data = read_and_process_file(data_file_path)
        if data is None:
            directory = os.path.dirname(data_file_path)
            list_files(directory, 'ascii')
    else:
        print("No data file path provided.")
        list_files(os.getcwd())

    # Check if the behaviour file path is provided and read the file
    behaviour_data = None
    if behaviour_file_path:
        behaviour_data = read_and_process_behaviour_file(behaviour_file_path)
        if behaviour_data is None:
            directory = os.path.dirname(behaviour_file_path)
            list_files(directory, 'csv')
    else:
        print("No behaviour file path provided.")
        list_files(os.getcwd())

    if data is not None:
        print("Data file read successfully.")
        print(f'''Data file path provided: {data_file_path}''')

    if behaviour_data is not None:
        print("Behaviour file read successfully.")
        print(f'''Behaviour file path provided: {behaviour_file_path}''')
   
    return data, behaviour_data

# Extracting and Processing Data for Plotting

This section handles the extraction, cleaning, and preparation of the data for plotting. It involves converting, parsing, and aligning the data based on the sample rates and a reference timestamp.

### Key Steps:
- **Data Conversion**:
  - Splits the raw data into metadata and numerical data for further processing.

- **DateTime Handling**:
  - Parses the `DateTime` values and aligns them with a reference timestamp.

- **Sample Rate Extraction**:
  - Retrieves sample rates from the metadata to properly downsample temperature and activity data.

- **Data Smoothing**:
  - Applies a Savitzky-Golay filter to smooth the pressure data for cleaner plots.

This ensures that the data is correctly formatted and aligned for accurate and meaningful visualizations.


In [ ]:
def extract_and_process_data(data, behaviour_data):
    # Convert the data to a list of strings to pass to find_skip_rows
    data_as_strings = data[0].tolist()
    
    # Parse the date strings into datetime objects
    probe_reference_timestamp = datetime.strptime(
        probe_date_time, '%m/%d/%Y %I:%M:%S %p')
    video_reference_timestamp = datetime.strptime(
        video_date_time, '%m/%d/%Y %I:%M:%S %p')
    
    print(f'''Probe timestamp: {probe_reference_timestamp}''')
    print(f'''Video timestamp: {video_reference_timestamp}''')

    skip_rows = find_skip_rows(data_as_strings)

    # Data is in 1 column, we need to split it into multiple columns where there is a ','
    data = data[0].str.split(',', expand=True)

    if skip_rows == -1:
        raise ValueError("Keyword 'Time' not found in the data.")

    # Split data into meta_data and numerical_data
    meta_data = data.iloc[:skip_rows]
    numerical_data = data.iloc[skip_rows:]

    # Remove the last 'None' column
    numerical_data = numerical_data.iloc[1:, :-1]

    # Assign headers to the columns for easier analysis
    numerical_data.columns = ['DateTime', 'Pressure', 'Temp', 'Activity']

    # Clean the DateTime column
    numerical_data['DateTime'] = numerical_data['DateTime'].str.strip()
    
    print(numerical_data['DateTime'].values[:5])

    # Convert all data columns to numeric
    numerical_data['Pressure'] = pd.to_numeric(
        numerical_data['Pressure'], errors='coerce')
    numerical_data['Temp'] = pd.to_numeric(
        numerical_data['Temp'], errors='coerce')
    numerical_data['Activity'] = pd.to_numeric(
        numerical_data['Activity'], errors='coerce')

    # Attempt to parse the DateTime column
    try:
        numerical_data['DateTime'] = pd.to_datetime(
            numerical_data['DateTime'], format='%m/%d/%Y %I:%M:%S.%f %p', errors='raise'
        )
    except ValueError as e:
        print(f'''Parsing failed: {e}''')
        numerical_data['DateTime'] = pd.to_datetime(
            numerical_data['DateTime'], errors='coerce')

    # Check for invalid DateTime values
    invalid_dates = numerical_data[numerical_data['DateTime'].isna()]
    if not invalid_dates.empty:
        print("Some DateTime values could not be parsed. Here are the first few invalid entries:")
        print(invalid_dates.head())

    # Calculate time difference between video and probe start
    video_time_diff = (video_reference_timestamp - probe_reference_timestamp).total_seconds()
    print(f'''Video start time difference: {video_time_diff} seconds''')

    # Track time difference for removed NaNs
    removed_nan_time_diff = 0

    # Print the number of NaN values at the start
    # nan_count_start = numerical_data['Pressure'].isna().sum()
    # print(
    #     f'''Total number of NaN values in 'Pressure' column: {nan_count_start}''')

    # Find the index of the first valid non-NaN value in 'Pressure'
    first_valid_index = numerical_data['Pressure'].first_valid_index()

    if first_valid_index is not None:
        # Adjust reference timestamp based on the first valid pressure data
        time_diff = numerical_data.loc[first_valid_index,
                                       'DateTime'] - probe_reference_timestamp
        new_reference_timestamp = probe_reference_timestamp + time_diff
        print(
            f'''Time difference between first valid pressure and reference timestamp: {time_diff}''')

        # Slice the data to include only from the first valid index onward
        numerical_data = numerical_data.loc[first_valid_index:].reset_index(
            drop=True)

        # Track the time difference caused by removing NaNs
        removed_nan_time_diff = time_diff.total_seconds()
        print(
            f'''Time difference due to NaN removal: {removed_nan_time_diff} seconds''')
    else:
        # No valid data was found
        raise ValueError(
            "No valid pressure data found. Cannot proceed with analysis.")

    # Calculate total time difference (video + NaN removal)
    total_time_diff = video_time_diff + removed_nan_time_diff
    print(f'''Total time difference (video + NaN): {total_time_diff} seconds''')

    # Re-zero the DateTime data based on the updated reference timestamp
    numerical_data['TimeSinceReference'] = (
        numerical_data['DateTime'] - new_reference_timestamp
    ).dt.total_seconds()

    # Convert DateTime to numeric for plotting
    numerical_data['TimeSinceReference'] = pd.to_numeric(
        numerical_data['TimeSinceReference'], errors='coerce')

    # Extract sample rates from the metadata
    col2_row = meta_data[meta_data[0].str.contains('Col 2:')].iloc[0]
    pressure_sample_rate = float(col2_row[4].split(':')[1].strip())

    col3_row = meta_data[meta_data[0].str.contains('Col 3:')].iloc[0]
    temp_sample_rate = float(col3_row[4].split(':')[1].strip())

    col4_row = meta_data[meta_data[0].str.contains('Col 4:')].iloc[0]
    activity_sample_rate = float(col4_row[4].split(':')[1].strip())

    # Calculate the interval for temperature and activity data based on their sample rates
    temp_interval = int(pressure_sample_rate / temp_sample_rate)
    activity_interval = int(pressure_sample_rate / activity_sample_rate)

    # Downsample temperature and activity data
    downsampled_temp = numerical_data.iloc[::temp_interval, [
        0, 2, 4]].reset_index(drop=True)
    downsampled_activity = numerical_data.iloc[::activity_interval, [
        0, 3, 4]].reset_index(drop=True)

    # Ensure the downsampled data has the correct headers
    downsampled_temp.columns = ['DateTime', 'Temp', 'TimeSinceReference']
    downsampled_activity.columns = [
        'DateTime', 'Activity', 'TimeSinceReference']

    # Create separate DataFrames for Pressure, Temperature, and Activity
    pressure_data = numerical_data[['TimeSinceReference', 'Pressure']].copy()
    temp_data = downsampled_temp[['TimeSinceReference', 'Temp']].copy()
    activity_data = downsampled_activity[[
        'TimeSinceReference', 'Activity']].copy()

    # Interpolate missing pressure data
    pressure_data['Pressure'].interpolate(method='linear', inplace=True)

    # Find the index of the first NaN from the end
    last_valid_index = pressure_data['Pressure'].last_valid_index()
    print(f'''Last valid index (non-NaN in Pressure): {last_valid_index}''')

    # Slice the DataFrame to include data only up to the last valid index
    pressure_data = pressure_data.loc[:last_valid_index]

    # Remove linear baseline trend
    detrended_data = detrend(pressure_data['Pressure'], type='linear')

    # Replace the original data with the detrended data
    pressure_data['Pressure'] = detrended_data

    # Apply Savitzky-Golay filter to smooth the 'Pressure' column
    smoothed_pressure = savgol_filter(
        pressure_data['Pressure'], window_length=35, polyorder=2)

    # Replace the 'Pressure' column with the smoothed data
    # pressure_data['Pressure'] = smoothed_pressure

    pressure_data.reset_index(drop=True, inplace=True)


    # Adjust the behaviour data if provided and is a dictionary
    if behaviour_data is not None:
        # Adjust the start and end times for each behavior
        for behaviour, instances in behaviour_data.items():
            adjusted_instances = []
            for instance_number, start_time, end_time, duration in instances:
                # Adjust start and end times by adding the total time difference
                adjusted_start_time = start_time + total_time_diff
                adjusted_end_time = end_time + total_time_diff

                # Check if the adjusted start or end time is negative, and skip those entries
                if adjusted_start_time >= 0 and adjusted_end_time >= 0:
                    # Keep the duration unchanged
                    adjusted_instances.append(
                        (instance_number, adjusted_start_time, adjusted_end_time, duration))
                else:
                    print(
                        f'''Skipping behavior instance {instance_number} due to negative start or end time.''')

            # Replace the original instances with the adjusted ones that have valid times
            behaviour_data[behaviour] = adjusted_instances

    print(pressure_data.head())
    print(smoothed_pressure[:10])     # instead of .head()

    return pressure_data, smoothed_pressure, temp_data, activity_data, numerical_data, new_reference_timestamp

# Peak Analysis Functions

This section includes functions to analyze different types of peaks in the data. 

### Key Functions:
- **`analyse_smaller_sleep_peaks`**: 
  - Identifies smaller peaks in sleep data and calculates the frequency of these peaks.
  
- **`analyse_sleeping_peaks`**: 
  - Detects and analyzes peaks and their pre-peak intervals in the smoothed pressure data, focusing on sleeping behavior.

- **Placeholder Functions**:
  - **`analyse_positive_peaks`** and **`analyse_sniffing_peaks`** are placeholders and can be implemented later to analyze other types of peaks.

These functions help in identifying and analyzing specific patterns in the data, particularly focusing on sleep-related peaks. The placeholders indicate where additional analysis functions can be added in the future.


In [168]:
def analyse_smaller_sleep_peaks(data, time_windows):
    # Assuming 'data' is a DataFrame with a 'Pressure' column
    pressure = data['Pressure']

    # Find peaks
    peaks, _ = find_peaks(pressure, prominence=1)

    # Calculate the times of peaks (assuming your DataFrame has a 'TimeSinceReference' column)
    peak_times = data['TimeSinceReference'].iloc[peaks]

    # Calculate intervals between peaks
    intervals = np.diff(peak_times)

    # Calculate frequency
    if len(intervals) > 0:
        average_interval = np.mean(intervals)
        frequency = 1 / average_interval
        return frequency, peaks, peak_times
    else:
        return None, None, None


def analyse_positive_peaks(time_windows):
    pass  # Placeholder function to be implemented later


def analyse_sniffing_peaks(time_windows):
    pass  # Placeholder function to be implemented later


def analyse_sleeping_peaks(time_windows, smoothed_pressure, pressure_data):
    results = []
    for start_time, end_time in time_windows:

        # Subset the data to only the current time window
        window_mask = (pressure_data['TimeSinceReference'] >= start_time) & (
            pressure_data['TimeSinceReference'] <= end_time)
        smoothed_pressure_window = smoothed_pressure[window_mask]
        pressure_data_window = pressure_data[window_mask]

        # Detecting peaks in the smoothed data (downward in original hence -smoothed)
        peaks, _ = find_peaks(-smoothed_pressure_window,
                              prominence=3, height=0)

        # Initialize an empty list to store the pre-peak indices
        pre_peak_indices = []

        # Loop through each detected peak
        for i in range(len(peaks)):
            peak = peaks[i]

            # Determine the range to search for the pre-peak
            if i == 0:
                search_range_start = 0
            else:
                search_range_start = peaks[i - 1]

            # Search only in the range between previous peak (or start) and current peak
            search_range_end = peak

            # Find local maxima in the specified range
            valid_maxima = argrelextrema(
                smoothed_pressure_window[search_range_start:search_range_end], np.greater)[0] + search_range_start

            if valid_maxima.size > 0:
                # Initialize variables to track the best pre-peak
                selected_pre_peak_index = valid_maxima[-1]
                min_distance_to_peak = float('inf')

                # Define an amplitude difference threshold relative to the peak
                amplitude_threshold = 0.5 * -smoothed_pressure_window[peak]

                for idx in valid_maxima:
                    distance_to_peak = peak - idx
                    amplitude_drop = smoothed_pressure_window[idx] - \
                        smoothed_pressure_window[peak]

                    # Check if this maxima is close enough to the peak and has a reasonable amplitude
                    if amplitude_drop >= amplitude_threshold:
                        if distance_to_peak < min_distance_to_peak:
                            min_distance_to_peak = distance_to_peak
                            selected_pre_peak_index = idx

                # Append the selected pre-peak index
                pre_peak_indices.append(selected_pre_peak_index)
            else:
                # Fallback to the highest maxima between peaks if no valid maxima are found
                if i > 0:
                    max_index = np.argmax(
                        smoothed_pressure_window[search_range_start:search_range_end]) + search_range_start
                    pre_peak_indices.append(max_index)
                else:
                    # Handle the case where there are no valid maxima and it's the first peak
                    selected_pre_peak_index = peaks[0]
                    pre_peak_indices.append(selected_pre_peak_index)

        # Ensure that both `peaks` and `pre_peak_indices` have the same length
        if len(peaks) != len(pre_peak_indices):
            peaks = peaks[:len(pre_peak_indices)]

        # Convert final lists to numpy arrays before appending
        peaks = np.array(peaks)
        pre_peak_indices = np.array(pre_peak_indices)

        # Collect times for peaks and their onset points
        peak_times = pressure_data_window['TimeSinceReference'].iloc[peaks]
        pre_peak_times = pressure_data_window['TimeSinceReference'].iloc[pre_peak_indices]

        if len(peak_times) != len(pre_peak_times):
            peak_times = peak_times[:len(pre_peak_times)]

        # Append results for the current window
        results.append((start_time, end_time, peaks, peak_times,
                       pre_peak_indices, pre_peak_times))

    return results

# Identifying Valid Periods

This function identifies valid periods in the data where the pressure exceeds a dynamically calculated threshold and contains a sufficient number of peaks. It returns a list of tuples representing the start and end times of these valid periods.


In [169]:
def identify_new_periods(start_time, end_time, peak_times, pressure_data):
    min_peaks = 50                  # Minimum number of peaks to consider a period valid
    interval_window = 10            # Number of recent intervals to calculate the median
    break_multiplier = 2.0          # Multiplier to determine a significant break

    # Convert peak_times to a sorted numpy array
    peak_times = np.array(peak_times)
    peak_times_filtered = peak_times[
        (peak_times >= start_time) & (peak_times <= end_time)
    ]
    peak_times_filtered.sort()

    if len(peak_times_filtered) < min_peaks:
        print("Not enough peaks in the window to form a valid period.")
        return []

    valid_periods = []
    current_period_start = peak_times_filtered[0]
    current_period_peaks = [peak_times_filtered[0]]
    recent_intervals = []

    for i in range(1, len(peak_times_filtered)):
        current_peak = peak_times_filtered[i]
        previous_peak = peak_times_filtered[i - 1]
        interval = current_peak - previous_peak

        # Update recent intervals
        recent_intervals.append(interval)
        if len(recent_intervals) > interval_window:
            recent_intervals.pop(0)

        # Calculate median interval
        median_interval = np.median(recent_intervals)

        # Check for significant break
        if interval > break_multiplier * median_interval:
            if len(current_period_peaks) >= min_peaks:
                period_end = previous_peak
                valid_periods.append((current_period_start, period_end))
                print(f'''Valid period detected: {current_period_start:.3f}s to {period_end:.3f}s with {len(current_period_peaks)} peaks''')
            else:
                print(f'''Period from {current_period_start:.3f}s to {previous_peak:.3f}s discarded: Only {len(current_period_peaks)} peaks''')

            # Start a new period
            current_period_start = current_peak
            current_period_peaks = [current_peak]
            recent_intervals = []
            continue  # Move to the next peak

        # Append the current peak to the ongoing period
        current_period_peaks.append(current_peak)

    # After iterating, check the last period
    if len(current_period_peaks) >= min_peaks:
        period_end = peak_times_filtered[-1]
        valid_periods.append((current_period_start, period_end))
        print(f'''Valid period detected: {current_period_start:.3f}s to {period_end:.3f}s with {len(current_period_peaks)} peaks''')
    else:
        print(f'''Last period from {current_period_start:.3f}s to {peak_times_filtered[-1]:.3f}s discarded: Only {len(current_period_peaks)} peaks''')

    return valid_periods


# Calculating Respiratory Metrics

This function calculates key respiratory metrics, such as inspiratory time, expiratory time, total cycle time, duty cycle, and respiratory drive based on identified peaks and troughs in the pressure data.

In [170]:
def bin_data_and_calculate_metrics(period_start_time, period_end_time, peak_times, trough_times, pressure_data, temp_data, activity_data):
    # Define bin edges based on the bin size, aligned with period_start_time
    bin_edges = np.arange(
        period_start_time, period_end_time + bin_size_sec, bin_size_sec)

    # Remove the last bin if it doesn't span the full bin size
    if (period_end_time - period_start_time) % bin_size_sec != 0:
        bin_edges = bin_edges[:-1]

    # Create a list to store metrics for each bin
    binned_metrics = []

    for i in range(len(bin_edges) - 1):
        # Define the start and end of the current bin
        bin_start = bin_edges[i]
        bin_end = bin_edges[i+1]

        # Select peaks and troughs within the current bin using boolean masks
        bin_peak_mask = (peak_times >= bin_start) & (peak_times < bin_end)
        bin_trough_mask = (trough_times >= bin_start) & (
            trough_times < bin_end)

        bin_peaks = peak_times[bin_peak_mask]
        bin_troughs = trough_times[bin_trough_mask]

        # Select temperature data within the current bin
        bin_temp_mask = (temp_data.index >= bin_start) & (
            temp_data.index < bin_end)
        bin_temps = temp_data[bin_temp_mask]

        # Select activity data within the current bin
        bin_activity_mask = (activity_data.index >= bin_start) & (
            activity_data.index < bin_end)
        bin_activity = activity_data[bin_activity_mask]

        # Skip bins that have insufficient data
        if len(bin_peaks) < 2 or len(bin_troughs) < 2:
            print(f'''Skipping bin {
                  i + 1} due to insufficient peaks or troughs.''')
            continue

        # Calculate respiratory metrics for the current bin
        # Note: If passing bin_peaks twice is intentional, keep as is; otherwise, adjust accordingly
        bin_metrics = calculate_respiratory_metrics(
            bin_peaks, bin_peaks, bin_troughs, pressure_data)
        bin_metrics['Bin_Start'] = bin_start
        bin_metrics['Bin_End'] = bin_end

        # Calculate temperature metrics within the bin
        if not bin_temps.empty:
            mean_temp = bin_temps['Temp'].mean()
            std_temp = bin_temps['Temp'].std()
            n_temp = bin_temps['Temp'].count()
            sem_temp = std_temp / np.sqrt(n_temp) if n_temp > 0 else np.nan
            bin_metrics['Mean_Temperature'] = mean_temp
            bin_metrics['SEM_Temperature'] = sem_temp
        else:
            bin_metrics['Mean_Temperature'] = np.nan
            bin_metrics['SEM_Temperature'] = np.nan

        # Calculate activity metrics within the bin
        if not bin_activity.empty:
            mean_activity = bin_activity['Activity'].mean()
            std_activity = bin_activity['Activity'].std()
            n_activity = bin_activity['Activity'].count()
            sem_activity = std_activity / \
                np.sqrt(n_activity) if n_activity > 0 else np.nan
            bin_metrics['Mean_Activity'] = mean_activity
            bin_metrics['SEM_Activity'] = sem_activity
        else:
            bin_metrics['Mean_Activity'] = np.nan
            bin_metrics['SEM_Activity'] = np.nan

        # Append metrics to the list
        binned_metrics.append(bin_metrics)

    # Convert list of metrics into a DataFrame
    binned_metrics_df = pd.DataFrame(binned_metrics)

    # Identify respiratory metrics columns
    respiratory_cols = [col for col in binned_metrics_df.columns if col not in [
        'Bin_Start', 'Bin_End', 'Mean_Temperature', 'SEM_Temperature', 'Mean_Activity', 'SEM_Activity']]

    # Create empty columns for separation
    binned_metrics_df['Empty_1'] = np.nan
    binned_metrics_df['Empty_2'] = np.nan

    # Define the desired column order
    # ['Bin_Start', 'Bin_End', respiratory data..., 'Empty_1', 'Mean_Temperature', 'SEM_Temperature', 'Empty_2', 'Mean_Activity', 'SEM_Activity']
    desired_columns = ['Bin_Start', 'Bin_End'] + respiratory_cols + [''] + \
        ['Mean_Temperature', 'SEM_Temperature'] + \
        [''] + ['Mean_Activity', 'SEM_Activity']

    # Reorder the DataFrame columns accordingly
    # Ensure all desired columns are present
    missing_cols = [
        col for col in desired_columns if col not in binned_metrics_df.columns]
    for col in missing_cols:
        binned_metrics_df[col] = np.nan  # Add missing columns with NaN values

    # Final column ordering
    binned_metrics_df = binned_metrics_df[desired_columns]

    return binned_metrics_df


# Function to calculate respiratory metrics from peak and trough data
def calculate_respiratory_metrics(period_peaks, peak_indices, trough_indices, pressure_data):
    # Ensure that every peak has a corresponding pre-peak
    if trough_indices.iloc[0] > peak_indices.iloc[0]:
        peak_indices = peak_indices.iloc[1:]
        period_peaks = period_peaks[1:]

    if peak_indices.iloc[-1] < trough_indices.iloc[-1]:
        trough_indices = trough_indices.iloc[:-1]

    # Calculate respiratory metrics using time values
    T_I = peak_indices.values - trough_indices.values
    T_E = trough_indices.values[1:] - peak_indices.values[:-1]
    T_TOT = T_I[:-1] + T_E
    duty_cycle = T_I[:-1] / T_TOT

    # Use the actual indices from peak_indices and trough_indices for calculating PI
    peak_indices_actual = peak_indices.index
    trough_indices_actual = trough_indices.index

    # Access pressure data using the actual integer indices
    peak_pressures = pressure_data.loc[peak_indices_actual, 'Pressure'].values
    trough_pressures = pressure_data.loc[trough_indices_actual,
                                         'Pressure'].values

    # Calculate pressure difference across the respiratory cycle using indices
    PI = peak_pressures - trough_pressures

    # Take the absolute difference to get the magnitude of change
    PI = np.abs(PI)
    respiratory_drive = PI / T_I

    peak_to_peak_times = np.diff(peak_indices.values)

    return {
        'T_I_mean': T_I.mean(),
        'T_E_mean': T_E.mean(),
        'T_TOT_mean': T_TOT.mean(),
        'Duty_Cycle_mean': duty_cycle.mean(),
        'Respiratory_Drive_mean': respiratory_drive.mean(),
        'Peak_to_Peak_mean': peak_to_peak_times.mean(),
        'Pressure Difference': PI.mean()
    }

# Plotting the nth Longest Behaviour

This function helps you plot the nth longest occurrence of a specific behavior from the data. It also allows you to restrict the behavior by duration and apply a buffer time before and after the behavior.


In [171]:
# Function to plot the nth longest behaviour
def plot_nth_longest_behaviour(behaviour_data, behaviour_to_plot,
                               numerical_data, buffer_time_before, buffer_time_after, restrict_duration, min_duration, max_duration, nth_longest):

    behaviour_instances = behaviour_data[behaviour_to_plot]

    # Check if duration restriction is needed
    if restrict_duration:
        # Filter instances within the defined duration range
        behaviour_instances = [
            instance for instance in behaviour_instances if min_duration <= instance[3] <= max_duration]
        if not behaviour_instances:
            print("No instances found within the specified range.")
            return None, None  # Exit if no instances match the criteria
    # Sort instances by duration in descending order
    sorted_instances = sorted(
        behaviour_instances, key=lambda x: x[3], reverse=True)

    # Adjust for zero-based index and check if the user-specified rank is valid
    nth_index = nth_longest - 1
    if nth_index < len(sorted_instances):
        instance_to_plot = sorted_instances[nth_index]
        print(f'''Plotting {nth_longest}-longest '{behaviour_to_plot}'...''')
    else:
        print("Not enough instances to select the specified rank.")
        return None, None

    # Extract details of the selected instance
    _, start_time, end_time, _ = instance_to_plot
    start_time -= buffer_time_before
    end_time += buffer_time_after
    time_windows = [(start_time, end_time)]

    return time_windows

# Function: `plot_data`

This function creates a comprehensive plot that visualizes pressure, temperature, and activity data over time. 

### Key Features:
- **Pressure Data:** Plotted with optional peaks highlighted.
- **Temperature Data:** Plotted on a secondary y-axis.
- **Activity Data:** Displayed as a histogram.
- **Valid Periods:** Marked with vertical lines for clear visualization of specific time intervals.

The function combines multiple data types into a single plot, making it easier to analyze relationships between pressure, temperature, and activity.

### Usage:
- **Inputs:**
  - `start_time`, `end_time`: Time range for plotting.
  - `pressure_data`, `temp_data`, `activity_data`: Dataframes containing respective data.
  - `peak_times`, `pre_peak_times`: Specific time points for peaks.
  - `valid_periods`: List of start and end times marking valid periods.
  
- **Output:** A plot displaying the data over the specified time range.


In [172]:
def plot_data(start_time, end_time, pressure_data, temp_data, activity_data, peak_times, pre_peak_times, valid_periods):
    percentage_offset = 0.05  # This means 5% above each peak
    
    # Plot the data
    fig, ax1 = plt.subplots(figsize=(14, 6))

    # Plot Pressure
    ax1.plot(pressure_data['TimeSinceReference'],
             pressure_data['Pressure'], color='k', label='Pressure', linewidth=0.8)
    if show_peaks:
        valid_peak_indices = pressure_data[pressure_data['TimeSinceReference'].isin(peak_times)].index
        ax1.scatter(pressure_data.loc[valid_peak_indices, 'TimeSinceReference'],
                    pressure_data.loc[valid_peak_indices, 'Pressure'] * (1 + percentage_offset),
                    color='m', marker='*', s=10, label='Peaks')

        valid_pre_peak_indices = pressure_data[pressure_data['TimeSinceReference'].isin(pre_peak_times)].index
        for idx in valid_pre_peak_indices:
                ax1.annotate('', 
                xy=(pressure_data.loc[idx, 'TimeSinceReference'], pressure_data.loc[idx, 'Pressure']), 
                xytext=(pressure_data.loc[idx, 'TimeSinceReference'], pressure_data.loc[idx, 'Pressure'] + 0.2), 
                arrowprops=dict(facecolor='gold', edgecolor='gold', arrowstyle='-|>', lw=0.7))
                
    # # Define a list of colors to cycle through
    # colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k']

    # # Iterate through the valid periods and apply a different color for each
    # for i, (period_start, period_end) in enumerate(valid_periods):
    #     print('''Period {i + 1}: {period_start} - {period_end}''')
    #     color = colors[i % len(colors)]  # Cycle through the colors list
    #     ax1.axvspan(period_start, period_end, color=color, alpha=0.1)
    
    ax1.set_xlabel('Time Since Reference (seconds)')
    ax1.set_ylabel('Pressure', color='b')
    ax1.tick_params(axis='y', labelcolor='b')
    ax1.yaxis.set_major_locator(plt.MaxNLocator(10))
    ax1.set_xlim(start_time, end_time)

    # Plot Temperature
    temp_y_max = temp_data['Temp'].max() * 1.01
    temp_y_min = temp_data['Temp'].min() * 0.97

    ax2 = ax1.twinx()
    ax2.set_ylim(temp_y_min, temp_y_max)
    ax2.plot(temp_data['TimeSinceReference'],
             temp_data['Temp'], color='r', label='Temp')
    ax2.set_ylabel('Temperature', color='r')
    ax2.tick_params(axis='y', labelcolor='r')

    num_bins = calculate_dynamic_bins(len(activity_data['TimeSinceReference']))

    # Plot Activity as a histogram
    act_y_max = activity_data['Activity'].max() * 4
    ax3 = ax1.twinx()
    ax3.set_ylim(0, act_y_max)
    ax3.spines['right'].set_position(('outward', 60))
    ax3.hist(activity_data['TimeSinceReference'], weights=activity_data['Activity'],
             bins=num_bins, color='g', alpha=0.4, label='Activity')
    ax3.set_ylabel('Activity', color='g')
    ax3.tick_params(axis='y', labelcolor='g')

    # Add legends
    fig.legend(loc='upper right', bbox_to_anchor=(
        1, 1), bbox_transform=ax1.transAxes)

    # Show the plot
    plt.title('Pressure, Activity, and Temperature over Time')
    print("Plotting the data...")
    plt.show()

In [173]:
def export_data(all_metrics, valid_peak_times_all, valid_pre_peak_times_all, output_filename='respiratory_metrics.xlsx'):
    # Create a Pandas Excel writer using XlsxWriter as the engine
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        # Write each period's metrics DataFrame to a separate sheet
        for period_name, metrics_df in all_metrics.items():
            # Use period_name as the sheet name
            metrics_df.to_excel(writer, sheet_name=period_name, index=False)
        
        # Create a DataFrame for valid peak and pre-peak times
        valid_peak_times_df = pd.DataFrame({
            'Valid_Peak_Times': valid_peak_times_all,
            'Valid_Pre_Peak_Times': valid_pre_peak_times_all
        })
        # Write valid peak times to a separate sheet
        valid_peak_times_df.to_excel(writer, sheet_name='Valid_Peak_Times', index=False)

    print(f'''Data has been successfully exported to {output_filename}''')

# Selecting Time Windows for Analysis

This function determines the time windows to be analyzed based on the specified behavior. If the behavior is not found in the data, it defaults to analyzing the entire dataset. If the behavior is present, it identifies the nth longest occurrence of that behavior, optionally applying duration restrictions and buffer times.

**Output**: 
- `time_windows`: A list of tuples representing the start and end times of the selected time windows.


In [174]:
def select_time_windows(behaviour_to_plot, behaviour_data, buffer_time_before, buffer_time_after, min_duration, reference_timestamp):
    # Ensure min_duration is a scalar value
    min_duration = min_duration[0] if isinstance(
        min_duration, tuple) else min_duration

    # Define the 90-minute cutoff time using Timedelta
    max_allowed_time = reference_timestamp + pd.Timedelta(minutes=90)

    # Calculate the time in seconds from the reference timestamp
    max_allowed_time_seconds = max_allowed_time.timestamp()

    # Check if the behavior to plot exists in the dictionary
    if behaviour_to_plot not in behaviour_data:
        return []

    # Get the list of tuples for the specified behavior
    behaviour_list = behaviour_data[behaviour_to_plot]

    # Filter out behaviors that start or end after the 90-minute mark
    valid_behaviours = [
        (instance, start, end, duration) for instance, start, end, duration in behaviour_list
        if start <= max_allowed_time_seconds and end <= max_allowed_time_seconds
    ]

    # Filter by minimum duration and handle NaN values
    valid_behaviours = [
        (instance, start, end, duration) for instance, start, end, duration in valid_behaviours
        if isinstance(duration, (float, int)) and not math.isnan(duration) and duration >= min_duration
    ]

    # Extract time windows (start time and end time)
    time_windows = [(start, end) for _, start, end, _ in valid_behaviours]

    return time_windows

# Analyzing Sleeping Peaks Across Time Windows

This function iterates over the specified time windows to analyze sleeping peaks. It calls `analyze_sleeping_peaks` for each window, using the smoothed pressure data to identify peaks and their characteristics within each window.

**Output**: 
- `results`: A list of peak analysis results for each time window.


In [175]:
def analyze_sleeping_peaks_for_windows(time_windows, smoothed_pressure, pressure_data):
    results = {}

    # Step 1: Analyze sleeping peaks for each time window
    for start_time, end_time in time_windows:
        window_key = f'''{start_time}-{end_time}'''
        print(f'''Analyzing window from {start_time} to {end_time}''')
        peaks_data = analyse_sleeping_peaks(
            [(start_time, end_time)], smoothed_pressure, pressure_data)
        results[window_key] = peaks_data

    return results

# Identifying Valid Periods and Calculating Metrics

This function processes the results from peak analysis to identify valid periods where the pressure data meets certain criteria. It then calculates respiratory metrics like inspiratory time, expiratory time, duty cycle, and respiratory drive for each valid period. The function also handles combining adjacent periods if they are closely spaced.

**Output**: 
- `valid_peak_times_all`: A list of all valid peak times.
- `valid_pre_peak_times_all`: A list of all valid pre-peak times.
- `updated_valid_periods`: A list of tuples representing the start and end times of each valid period.


In [176]:
def find_valid_periods(results, smoothed_data, pressure_data, temp_data, activity_data, time_windows):
    valid_peak_times_all = []
    valid_pre_peak_times_all = []
    updated_valid_periods = []
    all_metrics = {}  # Store metrics for all periods under each time window

    for window_start_time, window_end_time in time_windows:
        window_key = f'''{window_start_time}-{window_end_time}'''
        # Initialize storage for each time window's valid periods
        all_metrics[window_key] = {}

        # Retrieve results specific to the current time window
        window_results = results.get(window_key, [])
        
        if not window_results:
            print(f'''No results found for time window {window_key}. Skipping.''')
            continue

        for result in window_results:
            start_time, end_time, peaks, peak_times, pre_peak_indices, pre_peak_times = result
           
            valid_periods = identify_new_periods(
                start_time, end_time, peak_times, pressure_data)

            # Debug statement to check if valid periods were found
            if not valid_periods:
                print(
                    f'''No valid periods found for time window {window_key}. No peaks will be collected for this segment.''')
                continue

            for i, (period_start_time, period_end_time) in enumerate(valid_periods):
                # Only process periods within the current time window
                if not (window_start_time <= period_start_time <= window_end_time and window_start_time <= period_end_time <= window_end_time):
                    continue

                # Mask for peaks and pre-peaks within this valid period
                peak_period_mask = (peak_times >= period_start_time) & (
                    peak_times <= period_end_time)
                pre_peak_period_mask = (pre_peak_times >= period_start_time) & (
                    pre_peak_times <= period_end_time)

                period_peak_times = peak_times[peak_period_mask]
                period_trough_times = pre_peak_times[pre_peak_period_mask]
                period_peaks = peaks[peak_period_mask]

                # Fix for pre_peak_indices
                period_pre_peak_indices = [
                    pre_peak_indices[idx] for idx, flag in enumerate(pre_peak_period_mask) if flag]
                period_trough_indices = period_pre_peak_indices
                period_peak_indices = peaks[peak_period_mask]

                # Check for length mismatch between peaks and troughs
                if len(period_peak_times) > len(period_trough_times):
                    # Drop the first peak
                    period_peak_times = period_peak_times[1:]
                    period_peaks = period_peaks[1:]
                    period_peak_indices = period_peak_indices[1:]
                    
                elif len(period_trough_times) > len(period_peak_times):
                    # Drop the last trough
                    period_trough_times = period_trough_times[:-1]
                    period_trough_indices = period_trough_indices[:-1]

                # Calculate and store respiratory metrics for this valid period
                binned_metrics_df = bin_data_and_calculate_metrics(
                    period_start_time, period_end_time, period_peak_times, period_trough_times, pressure_data, temp_data, activity_data)

                # Store metrics for this valid period under the current time window
                period_name = f'''Period_{i+1}_{period_start_time}-{period_end_time}'''
                all_metrics[window_key][period_name] = binned_metrics_df

                # Store valid peak and trough times
                valid_peak_times_all.extend(period_peak_times)
                valid_pre_peak_times_all.extend(period_trough_times)

                # Add the valid period to the updated periods
                updated_valid_periods.append(
                    (period_start_time, period_end_time))

    return valid_peak_times_all, valid_pre_peak_times_all, updated_valid_periods, all_metrics

In [177]:
def export_full_time_range_plot(pressure_data, temp_data, activity_data,
                                valid_peak_times_all, valid_pre_peak_times_all,
                                min_time, max_time, behaviour_to_plot, full_trace_folder, file_base):

    # Filter data to only include values within min_time and max_time
    pressure_filtered = pressure_data[(pressure_data['TimeSinceReference'] >= min_time) &
                                      (pressure_data['TimeSinceReference'] <= max_time)]

    temp_filtered = temp_data[(temp_data['TimeSinceReference'] >= min_time) &
                              (temp_data['TimeSinceReference'] <= max_time)]

    activity_filtered = activity_data[(activity_data['TimeSinceReference'] >= min_time) &
                                      (activity_data['TimeSinceReference'] <= max_time)]

    # Ensure peak lists are NumPy arrays before filtering
    valid_peak_times_all = np.array(
        valid_peak_times_all, dtype=np.float64)  # Force numerical type
    valid_pre_peak_times_all = np.array(
        valid_pre_peak_times_all, dtype=np.float64)

    # Now apply filtering
    valid_peak_times = valid_peak_times_all[(valid_peak_times_all >= min_time) &
                                            (valid_peak_times_all <= max_time)]

    valid_pre_peak_times = valid_pre_peak_times_all[(valid_pre_peak_times_all >= min_time) &
                                                    (valid_pre_peak_times_all <= max_time)]

    # **Save Plotly Interactive HTML**
    fig = go.Figure()

    # Plot Pressure
    fig.add_trace(go.Scatter(x=pressure_filtered['TimeSinceReference'],
                             y=pressure_filtered['Pressure'],
                             mode='lines',
                             name='Pressure',
                             line=dict(color='black', width=1)))

    # Plot Temperature on secondary y-axis
    fig.add_trace(go.Scatter(x=temp_filtered['TimeSinceReference'],
                             y=temp_filtered['Temp'],
                             mode='lines',
                             name='Temperature',
                             line=dict(color='red', width=1),
                             yaxis='y2'))

    # Plot Activity
    fig.add_trace(go.Scatter(x=activity_filtered['TimeSinceReference'],
                             y=activity_filtered['Activity'],
                             mode='lines',
                             name='Activity',
                             line=dict(color='green', width=1, dash='dot')))

    # Peaks and Pre-Peaks
    fig.add_trace(go.Scatter(x=valid_peak_times,
                             y=pressure_data.loc[pressure_data['TimeSinceReference'].isin(
                                 valid_peak_times), 'Pressure'],
                             mode='markers',
                             name='Peaks',
                             marker=dict(color='magenta', size=6, symbol='cross')))

    fig.add_trace(go.Scatter(x=valid_pre_peak_times,
                             y=pressure_data.loc[pressure_data['TimeSinceReference'].isin(
                                 valid_pre_peak_times), 'Pressure'],
                             mode='markers',
                             name='Pre-Peaks',
                             marker=dict(color='gold', size=6, symbol='triangle-up')))

    # Set layout
    fig.update_layout(
        title=f'''{behaviour_to_plot}: Full Time Range {min_time:.2f} to {max_time:.2f}''',
        xaxis_title='Time Since Reference (seconds)',
        yaxis_title='Pressure',
        yaxis2=dict(title='Temperature', overlaying='y', side='right', color='red'),
        legend=dict(orientation="h", yanchor="bottom",
                    y=1.02, xanchor="right", x=1),
        height=600,
        hovermode='x unified',
        xaxis=dict(range=[min_time, max_time])  # Limit x-axis to full range
    )

    # Define file paths
    save_path_html = os.path.join(
        full_trace_folder, f'''{file_base}_full_trace_{min_time:.2f}_to_{max_time:.2f}.html''')
    save_path_svg = os.path.join(
        full_trace_folder, f'''{file_base}_full_trace_{min_time:.2f}_to_{max_time:.2f}.svg''')

    # Save as HTML (interactive)
    fig.write_html(save_path_html)
    print(f'''Saved full trace plot as HTML:\n  - {save_path_html}''')

    # **Save Matplotlib Static SVG**
    # Extract data for Matplotlib
    x_data = pressure_filtered['TimeSinceReference']
    y_data = pressure_filtered['Pressure']

    # Create Matplotlib figure
    mpl_fig, ax1 = plt.subplots(figsize=(12, 6))

    # Plot Pressure
    ax1.plot(x_data, y_data, color='black', label="Pressure")
    ax1.set_xlabel("Time Since Reference (seconds)")
    ax1.set_ylabel("Pressure", color='black')

    # Plot Temperature on secondary y-axis
    ax2 = ax1.twinx()
    ax2.plot(temp_filtered['TimeSinceReference'],
             temp_filtered['Temp'], color='red', label="Temperature")
    ax2.set_ylabel("Temperature", color='red')

    # Plot Activity
    ax1.plot(activity_filtered['TimeSinceReference'], activity_filtered['Activity'],
             color='green', linestyle='dotted', label="Activity")

    # Plot Peaks and Pre-Peaks
    if len(valid_peak_times) > 0:
        ax1.scatter(valid_peak_times, pressure_data.loc[pressure_data['TimeSinceReference'].isin(valid_peak_times), 'Pressure'],
                    color='magenta', marker='x', label="Peaks")

    if len(valid_pre_peak_times) > 0:
        ax1.scatter(valid_pre_peak_times, pressure_data.loc[pressure_data['TimeSinceReference'].isin(valid_pre_peak_times), 'Pressure'],
                    color='gold', marker='^', label="Pre-Peaks")

    # Final plot settings
    ax1.legend(loc="upper left")
    ax1.grid(True)

    # Save as SVG
    mpl_fig.savefig(save_path_svg, format="svg", bbox_inches='tight')
    plt.close(mpl_fig)  # Close figure to prevent memory issues

    print(
        f'''Finished saving full trace interactive HTML and static SVG plots to {full_trace_folder}''')

In [178]:
def export_behavior_images_interactive(time_windows, pressure_data, temp_data, activity_data,
                                       valid_peak_times_all, valid_pre_peak_times_all, behaviour_to_plot, html_save_folder, svg_save_folder, file_base):
    # Loop through each behavior time window
    for i, (start_time, end_time) in enumerate(time_windows):
        # Define time margins
        start_time_margin = start_time
        end_time_margin = end_time

        # Filter data for the current time window
        behavior_segment = pressure_data[
            (pressure_data['TimeSinceReference'] >= start_time_margin) &
            (pressure_data['TimeSinceReference'] <= end_time_margin)
        ]
        temp_segment = temp_data[
            (temp_data['TimeSinceReference'] >= start_time_margin) &
            (temp_data['TimeSinceReference'] <= end_time_margin)
        ]
        activity_segment = activity_data[
            (activity_data['TimeSinceReference'] >= start_time_margin) &
            (activity_data['TimeSinceReference'] <= end_time_margin)
        ]

        # Convert lists to numpy arrays
        valid_peak_times_all = np.array(valid_peak_times_all)
        valid_pre_peak_times_all = np.array(valid_pre_peak_times_all)

        # Filter peaks and pre-peaks within the window
        valid_peak_times = valid_peak_times_all[
            (valid_peak_times_all >= start_time_margin) & (
                valid_peak_times_all <= end_time_margin)
        ]
        valid_pre_peak_times = valid_pre_peak_times_all[
            (valid_pre_peak_times_all >= start_time_margin) & (
                valid_pre_peak_times_all <= end_time_margin)
        ]

        # Check if there's valid data before proceeding
        if behavior_segment.empty or temp_segment.empty or activity_segment.empty:
            print(
                f'''No data found for behavior from {start_time_margin} to {end_time_margin}. Skipping.''')
            continue

        # Downsample Pressure Data (from 500 Hz to 50 Hz, keep every 10th point)
        behavior_segment = behavior_segment.iloc[::10].copy()

        # **Save Plotly Interactive HTML**
        fig = go.Figure()

        # Plot Pressure
        fig.add_trace(go.Scatter(x=behavior_segment['TimeSinceReference'],
                                 y=behavior_segment['Pressure'],
                                 mode='lines',
                                 name='Pressure',
                                 line=dict(color='black', width=1)))

        # Plot Temperature on secondary y-axis
        fig.add_trace(go.Scatter(x=temp_segment['TimeSinceReference'],
                                 y=temp_segment['Temp'],
                                 mode='lines',
                                 name='Temperature',
                                 line=dict(color='red', width=1),
                                 yaxis='y2'))

        # Plot Activity
        fig.add_trace(go.Scatter(x=activity_segment['TimeSinceReference'],
                                 y=activity_segment['Activity'],
                                 mode='lines',
                                 name='Activity',
                                 line=dict(color='green', width=1, dash='dot')))

        # Peaks and Pre-Peaks
        fig.add_trace(go.Scatter(x=valid_peak_times,
                                 y=pressure_data.loc[pressure_data['TimeSinceReference'].isin(
                                     valid_peak_times), 'Pressure'],
                                 mode='markers',
                                 name='Peaks',
                                 marker=dict(color='magenta', size=6, symbol='cross')))

        fig.add_trace(go.Scatter(x=valid_pre_peak_times,
                                 y=pressure_data.loc[pressure_data['TimeSinceReference'].isin(
                                     valid_pre_peak_times), 'Pressure'],
                                 mode='markers',
                                 name='Pre-Peaks',
                                 marker=dict(color='gold', size=6, symbol='triangle-up')))

        # Layout settings
        fig.update_layout(
            title=f'''{behaviour_to_plot} from {start_time_margin:.2f} to {end_time_margin:.2f}''',
            xaxis_title='Time Since Reference (seconds)',
            yaxis_title='Pressure',
            yaxis2=dict(title='Temperature', overlaying='y',
                        side='right', color='red'),
            legend=dict(orientation="h", yanchor="bottom",
                        y=1.02, xanchor="right", x=1),
            height=600,
            hovermode='x unified',
        )

        # Save HTML
        save_path_html = os.path.join(
            html_save_folder, f'''{behaviour_to_plot}_behavior_{i}_from_{start_time:.2f}_to_{end_time:.2f}.html''')
        fig.write_html(save_path_html)
        print(f'''Saved HTML plot: {save_path_html}''')

        # **Save Matplotlib Static SVG**
        save_path_svg = os.path.join(
            svg_save_folder, f'''{behaviour_to_plot}_behavior_{i}_from_{start_time:.2f}_to_{end_time:.2f}.svg''')

        # Extract data for Matplotlib
        x_data = behavior_segment['TimeSinceReference']
        y_data = behavior_segment['Pressure']

        # Create Matplotlib figure
        mpl_fig, ax1 = plt.subplots(figsize=(12, 6))

        # Plot Pressure
        ax1.plot(x_data, y_data, color='black', label="Pressure")
        ax1.set_xlabel("Time Since Reference (seconds)")
        ax1.set_ylabel("Pressure", color='black')

        # Plot Temperature on secondary y-axis
        ax2 = ax1.twinx()
        ax2.plot(temp_segment['TimeSinceReference'],
                 temp_segment['Temp'], color='red', label="Temperature")
        ax2.set_ylabel("Temperature", color='red')

        # Plot Activity
        ax1.plot(activity_segment['TimeSinceReference'], activity_segment['Activity'],
                 color='green', linestyle='dotted', label="Activity")

        # Plot Peaks and Pre-Peaks
        if len(valid_peak_times) > 0:
            ax1.scatter(valid_peak_times, pressure_data.loc[pressure_data['TimeSinceReference'].isin(valid_peak_times), 'Pressure'],
                        color='magenta', marker='x', label="Peaks")

        if len(valid_pre_peak_times) > 0:
            ax1.scatter(valid_pre_peak_times, pressure_data.loc[pressure_data['TimeSinceReference'].isin(valid_pre_peak_times), 'Pressure'],
                        color='gold', marker='^', label="Pre-Peaks")

        # Final plot settings
        ax1.legend(loc="upper left")
        ax1.grid(True)

        # Save as SVG
        mpl_fig.savefig(save_path_svg, format="svg", bbox_inches='tight')
        plt.close(mpl_fig)  # Close figure to prevent memory issues

        print(f'''Saved SVG plot: {save_path_svg}''')

    print(
        f'''Finished saving all interactive HTML and static SVG plots to {html_save_folder} and {svg_save_folder}''')

In [179]:
def export_sleep_data_to_excel(summary_data, all_metrics, data_file_path):
    try:
        # Convert summary data to a DataFrame
        summary_df = pd.DataFrame(summary_data)
        
        # Extract the base name without extension
        file_base = os.path.splitext(os.path.basename(data_file_path))[0]

        # Define the folder for MRP analysis
        extracted_data_folder = 'extracted_data'

        # Define the analysis folder path (this is the only folder to be created)
        analysis_folder = os.path.join(extracted_data_folder, f'''{file_base}_MRP_analysis''')

        # Ensure the analysis folder exists
        os.makedirs(analysis_folder, exist_ok=True)

        # Get current date in YYYYMMDD format
        current_date = datetime.now().strftime('%Y%m%d')

        # Define the path for saving the Excel file with date
        excel_filename = f'''{file_base}_MRP_analysis_{
            current_date}_sleep_analysis.xlsx'''
        excel_path = os.path.join(analysis_folder, excel_filename)

        # Initialize the Excel writer
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            # Export summary data
            summary_df.to_excel(writer, sheet_name='Summary Data', index=False)

            # Export each time window and its periods into separate sheets
            for window, periods in all_metrics.items():
                # Create a list to hold rows for each time window
                window_data = []
                for period_index, (period_name, metrics_df) in enumerate(periods.items(), start=1):
                    # Add a row identifying the valid period (e.g., "Valid Period 1", "Valid Period 2")
                    window_data.append([f'''Valid Period {period_index}'''])

                    # Add column headers for the period data
                    window_data.append(metrics_df.columns.tolist())

                    # Append the actual data for this period
                    window_data.extend(metrics_df.values.tolist())

                    # Add an empty row between periods for clarity
                    window_data.append([])

                # Convert the window data into a DataFrame for export
                window_df = pd.DataFrame(window_data)

                # Clean the sheet name to be Excel compliant (max 31 chars, no special characters)
                clean_window = ''.join(c if c.isalnum() or c in (
                    ' ', '_') else '_' for c in window)
                # Excel sheet name max length is 31
                clean_window = clean_window[:31]

                # Export to Excel
                window_df.to_excel(
                    writer, sheet_name=clean_window, index=False, header=False)

        print(f'''Data has been successfully exported to {excel_path}''')

    except Exception as e:
        print(f'''Failed to export data to Excel: {e}''')
        # Continue execution without stopping

In [180]:
def create_summary_data(valid_peak_times_all, updated_valid_periods, time_windows):
    summary_data = []

    for window_start_time, window_end_time in time_windows:
        # Add the overall time window to the summary
        summary_data.append({
            'Start Time': window_start_time,
            'End Time': window_end_time,
            'Description': 'Overall Time Window'
        })

        # Add the valid periods within the time window
        for valid_start, valid_end in updated_valid_periods:
            if window_start_time <= valid_start <= window_end_time and window_start_time <= valid_end <= window_end_time:
                num_valid_peaks = len(
                    [peak for peak in valid_peak_times_all if valid_start <= peak <= valid_end])
                summary_data.append({
                    'Start Time': valid_start,
                    'End Time': valid_end,
                    'Number of Peaks': num_valid_peaks,
                    'Description': 'Valid Period'
                })

    return summary_data

# Customizable Parameters

Below are some parameters that you can easily change to customize the output without needing to modify the actual code.

- **File Paths**:
  - `data_file_path`: Path to your main data file.
  - `behaviour_file_path`: Path to your behavior data file.

- **Date and Time**:
  - `date_time`: Specify the date and time in the format `Month/Day/Year Hour:Minute:Second AM/PM`.

- **Behavior to Plot**:
  - `behaviour_to_plot`: Specify the behavior you want to focus on. Use `None` to plot the entire dataset.

- **Time windows**
  - `time_windows`: Set of time windows set by the user to look at instead of using behaviours

- **Buffer Times**:
  - `buffer_time_before`: Time in seconds to include before the behavior starts.
  - `buffer_time_after`: Time in seconds to include after the behavior ends.

- **Show Peaks**:
  - `show_peaks`: Set to `True` to display peaks and pre-peaks in the pressure data, or `False` to hide them.

- **Nth Longest Behavior**:
  - `nth_longest`: Select which instance of the behavior to plot (1 for the longest, 2 for the second longest, etc.).

- **Duration Restrictions**:
  - `restrict_duration`: Set to `True` to restrict the behavior to a specific duration range.
  - `min_duration` and `max_duration`: Define the minimum and maximum duration (in seconds) for the behavior.

- **Bin Size**:
  - `bin_size_sec`: Bin size for exporting the data in the periods.


In [181]:

# data_file_path = 'data/Day 2 (25-04-24) Pro - this is the NaNs file/B1 virgin Day 2 (25-04-24) ponemah.csv'

# # Modify this to the correct path for your behavior file
# behaviour_file_path = 'data/Day 2 (25-04-24) Pro - this is the NaNs file/B1 virgin Day 2 (25-04-24).csv'

# # Specify the date and time of interest
# # Format: Month/Day/Year Hour:Minute:Second AM/PM
# probe_date_time = '04/25/2024 9:45:09 AM'
# video_date_time = '04/25/2024 9:38:10 AM'

In [182]:
# File paths to the data files
# Modify this to the correct path for your data file
from pathlib import Path

folder = Path('/home/michael/code/pressure_analysis/data/B5/PM/Pro')
    
# Modify this to the correct path for your data file
data_file_path = folder / 'B5 Pro NIGHT 11-01-2025 Ponemah.csv'

# Modify this to the correct path for your behavior file
behaviour_file_path = folder / 'B5 Pro NIGHT 11-01-2025 BORIS.csv'

# Specify the date and time of interest
# Format: Month/Day/Year Hour:Minute:Second AM/PM
probe_date_time = '01/11/2025 5:05:09 PM'
video_date_time = '01/11/2025 5:05:09 PM'

# Specify the behavior you want to plot
# Set to behaviour_to_plot 'None' to plot the entire trace or specify the behavior of interest
behaviour_to_plot = 'Time spent sleeping'

# Set time_windows to 'None' to plot the behaviour of interest or specify the time windows of interest

# Define buffer times around the behavior
buffer_time_before = 60  # Time in seconds before the behavior starts
buffer_time_after = 60  # Time in seconds after the behavior ends

# Toggle whether to show peaks in the pressure data
show_peaks = True  # Set to 'False' if you do not want to display peaks

# Select which occurrence of the behavior to plot
nth_longest = 2  # 1 for the longest, 2 for the second longest, etc.

# Restrict the duration of the behavior
restrict_duration = True  # Set to 'False' to ignore duration restrictions

# Minimum duration of the behavior in seconds
min_duration = 30  # Set minimum duration to 30 seconds

# Maximum duration in seconds (use 'float('inf')' for no limit)
max_duration = float('inf')

# Bin size for the exported data in seconds
bin_size_sec = 10

In [183]:
def create_folders_for_graphs(data_file_path):
    # Extract the base filename without extension and directory
    file_base = os.path.splitext(os.path.basename(data_file_path))[0]
    
    save_images_folder = 'behaviour_images'

    # Create a folder to save the HTML files and svg images
    html_save_folder = os.path.join(save_images_folder, file_base)
    svg_save_folder = os.path.join(html_save_folder, 'Graphs for illustrator')
    full_trace_folder = os.path.join(html_save_folder, 'Full plot')  


    # Create folders (if they don't exist)
    os.makedirs(html_save_folder, exist_ok=True)
    os.makedirs(svg_save_folder, exist_ok=True)
    os.makedirs(full_trace_folder, exist_ok=True)
    
    return html_save_folder, svg_save_folder, full_trace_folder, file_base

# Analysis and Plotting Process

This script follows a sequence of steps to analyze and plot behavioral data:

1. **Select Time Windows**: Identifies relevant time windows based on the behavior to analyze.
2. **Analyze Sleeping Peaks**: Detects peaks within the selected time windows.
3. **Identify Valid Periods and Calculate Metrics**: Filters valid periods and computes respiratory metrics.
4. **Plot the Data**: Visualizes the analyzed data, showing pressure, temperature, and activity over time.


In [184]:
from typing import List, Tuple
from pathlib import Path
import pandas as pd
import numpy as np


def export_pressure_segments(
    pressure_data: pd.DataFrame,
    smoothed_pressure: np.ndarray,
    time_windows: List[Tuple[pd.Timestamp, pd.Timestamp]],
    output_folder: Path,
    base_filename: str = "pressure_segment"
):
    """Export raw and smoothed pressure data for each time window to separate CSV files."""
    pressure_data = pressure_data.copy()
    pressure_data['SmoothedPressure'] = smoothed_pressure

    output_folder.mkdir(parents=True, exist_ok=True)

    for i, (start, end) in enumerate(time_windows):
        segment = pressure_data[
            (pressure_data['TimeSinceReference'] >= start) &
            (pressure_data['TimeSinceReference'] <= end)
        ].copy()

        if not segment.empty:
            filename = output_folder / f"{base_filename}_window{i+1}.csv"
            segment.to_csv(filename, index=False)
            print(f"Saved window {i+1} to {filename}")
        else:
            print(f"Window {i+1} is empty and was skipped.")

In [185]:
# Step 0: Retrieve data
from pathlib import Path
data, behaviour_data = retireve_data()

# Step 1: Extract and process data
pressure_data, smoothed_pressure, temp_data, activity_data, numerical_data, new_reference_timestamp = extract_and_process_data(
    data, behaviour_data)

# Step 2: Select time windows
time_windows = select_time_windows(behaviour_to_plot, behaviour_data,
                                   buffer_time_before, buffer_time_after, min_duration, new_reference_timestamp)

print(pressure_data.columns)


export_pressure_segments(
    pressure_data=pressure_data,
    smoothed_pressure=smoothed_pressure,
    time_windows=time_windows,
    output_folder=Path("pressure_segment_exports")  # folder, not file
)


# Step 3: Analyze sleeping peaks
results = analyze_sleeping_peaks_for_windows(
    time_windows, smoothed_pressure, pressure_data)

# start_time, end_time, peaks, peak_times, pre_peak_indies, pre_peak_times = results[0]

all_peak_times = []
all_pre_peak_times = []

for window_key, window_results in results.items():
    for result in window_results:
        # Unpack the result tuple
        _, _, _, peak_times, _, pre_peak_times = result
        all_peak_times.extend(peak_times)
        all_pre_peak_times.extend(pre_peak_times)

# Step 4: Identify valid periods and calculate metrics
valid_peak_times_all, valid_pre_peak_times_all, updated_valid_periods, all_metrics = find_valid_periods(
    results, smoothed_pressure, pressure_data, temp_data, activity_data, time_windows)

# Step 5: Create summary and detailed data for exporting
summary_data = create_summary_data(
    valid_peak_times_all, updated_valid_periods, time_windows)

max_time = pressure_data['TimeSinceReference'].max()
min_time = pressure_data['TimeSinceReference'].min()

# # Remove this for now
# plot_data(pressure_data['TimeSinceReference'].min(), pressure_data['TimeSinceReference'].max(), pressure_data, temp_data,
#           activity_data, all_peak_times, all_pre_peak_times, updated_valid_periods)

# Make folders for exporting graphs
html_save_folder, svg_save_folder, full_trace_folder, file_base = create_folders_for_graphs(
    data_file_path)

# Step 6: Export full time range plot
export_full_time_range_plot(pressure_data, temp_data, activity_data,
                                valid_peak_times_all, valid_pre_peak_times_all, 
                            min_time, max_time, behaviour_to_plot, full_trace_folder, file_base)

print("Exported full time range plot")

# Step 7: Export images of the behavior
export_behavior_images_interactive(time_windows, pressure_data, temp_data, activity_data,
                                   valid_peak_times_all, valid_pre_peak_times_all, behaviour_to_plot, html_save_folder, svg_save_folder, file_base)

# Step 8: Export data to Excel
export_sleep_data_to_excel(summary_data, all_metrics, data_file_path)

Data file read successfully.
Data file path provided: /home/michael/code/pressure_analysis/data/B5/PM/Pro/B5 Pro NIGHT 11-01-2025 Ponemah.csv
Behaviour file read successfully.
Behaviour file path provided: /home/michael/code/pressure_analysis/data/B5/PM/Pro/B5 Pro NIGHT 11-01-2025 BORIS.csv
Probe timestamp: 2025-01-11 17:05:09
Video timestamp: 2025-01-11 17:05:09
Video start time difference: 0.0 seconds
Time difference between first valid pressure and reference timestamp: -1 days +23:54:39
Time difference due to NaN removal: -321.0 seconds
Total time difference (video + NaN): -321.0 seconds


/tmp/ipykernel_52566/532347364.py:140: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





Last valid index (non-NaN in Pressure): 5406499
   TimeSinceReference  Pressure
0               0.000 -5.403704
1               0.002 -3.895004
2               0.004 -2.504904
3               0.006 -1.269004
4               0.008 -0.254904
[ 0.89586888 -0.10979952 -1.03012589 -1.86511022 -2.61475251 -3.27905276
 -3.85801098 -4.35162715 -4.75990129 -5.08283338]
Index(['TimeSinceReference', 'Pressure'], dtype='object')
Saved window 1 to pressure_segment_exports/pressure_segment_window1.csv
Saved window 2 to pressure_segment_exports/pressure_segment_window2.csv
Saved window 3 to pressure_segment_exports/pressure_segment_window3.csv
Saved window 4 to pressure_segment_exports/pressure_segment_window4.csv
Saved window 5 to pressure_segment_exports/pressure_segment_window5.csv
Saved window 6 to pressure_segment_exports/pressure_segment_window6.csv
Saved window 7 to pressure_segment_exports/pressure_segment_window7.csv
Saved window 8 to pressure_segment_exports/pressure_segment_window8.csv
Sav

/tmp/ipykernel_52566/717053578.py:137: RuntimeWarning:

invalid value encountered in divide



Valid period detected: 7359.984s to 7510.644s with 538 peaks
Valid period detected: 7510.900s to 7523.800s with 59 peaks
Period from 7524.068s to 7525.412s discarded: Only 9 peaks
Valid period detected: 7525.776s to 7542.084s with 86 peaks
Period from 7542.458s to 7546.544s discarded: Only 22 peaks
Valid period detected: 7546.878s to 7649.176s with 404 peaks
Period from 8190.474s to 8191.306s discarded: Only 9 peaks
Period from 8191.528s to 8194.964s discarded: Only 33 peaks
Period from 8195.304s to 8195.822s discarded: Only 5 peaks
Period from 8196.092s to 8198.720s discarded: Only 26 peaks
Valid period detected: 8198.922s to 8207.022s with 70 peaks
Period from 8207.936s to 8208.818s discarded: Only 7 peaks
Valid period detected: 8209.128s to 8215.668s with 57 peaks
Period from 8216.254s to 8217.798s discarded: Only 13 peaks
Period from 8219.058s to 8222.966s discarded: Only 16 peaks
Period from 8223.252s to 8224.146s discarded: Only 3 peaks
Valid period detected: 8226.038s to 8236.57

/tmp/ipykernel_52566/717053578.py:137: RuntimeWarning:

invalid value encountered in divide

/tmp/ipykernel_52566/717053578.py:137: RuntimeWarning:

invalid value encountered in divide



Valid period detected: 8979.634s to 9177.602s with 634 peaks
Valid period detected: 9250.296s to 9294.040s with 141 peaks
Valid period detected: 9360.204s to 9470.746s with 396 peaks
Period from 9471.274s to 9476.712s discarded: Only 14 peaks
Period from 9485.016s to 9490.694s discarded: Only 8 peaks
Period from 9492.482s to 9493.068s discarded: Only 3 peaks
Period from 9493.862s to 9496.652s discarded: Only 16 peaks
Period from 9498.350s to 9502.270s discarded: Only 9 peaks
Last period from 9503.386s to 9505.768s discarded: Only 9 peaks
Period from 9998.596s to 10002.940s discarded: Only 28 peaks
Period from 10003.602s to 10005.090s discarded: Only 6 peaks
Period from 10005.812s to 10012.530s discarded: Only 26 peaks
Valid period detected: 10013.144s to 10024.428s with 54 peaks
Period from 10026.024s to 10030.370s discarded: Only 30 peaks
Period from 10031.466s to 10036.446s discarded: Only 28 peaks
Period from 10036.780s to 10037.286s discarded: Only 3 peaks
Period from 10040.248s to